# Notebook 3 — Fine-Tune YOLOv8 and Export to ONNX

**Use case:** Manufacturing Visual Quality Inspection

An assembly-line camera captures images of PCB components. Your fine-tuned model
classifies each image into one of four quality categories — flagging defects before
they ship to customers.

**What you will do:**
1. Build a 4-class dataset: `contamination`, `crack`, `pass`, `scratch` with strong augmentation.
2. Fine-tune YOLOv8n-cls on that dataset (CPU, ~5–10 min).
3. Export the best checkpoint to ONNX format.
4. Upload the ONNX model to your S3 bucket so it can be served by KServe.

> ⏱ **Timing:** Training 30 epochs on CPU typically takes 5–10 minutes.

---

## Why fine-tuning is needed

| Problem | Fix |
|---|---|
| Pre-trained model has no concept of PCB defects | Fine-tune on real PCB defect seed images |
| Small dataset → underfitting | **60 images/class** via strong augmentation |
| Class label index mismatch | CLASSES list explicitly matches YOLOv8 **alphabetical sort** |
| Weak augmentation | Rotation, perspective warp, noise, colour jitter |

---

## Class design

| Class | Meaning | Business impact if missed |
|---|---|---|
| `contamination` | Liquid, flux, or oil on PCB | Short circuit in field → recall |
| `crack` | Structural crack in solder joint or board | Intermittent failure → warranty claim |
| `pass` | Clean, defect-free board | Ships to customer ✅ |
| `scratch` | Surface scratch cutting traces | Signal integrity failure |

> **ONNX output order (alphabetical):** index 0 = `contamination`, 1 = `crack`, 2 = `pass`, 3 = `scratch`

In [ ]:
# ── Cell 0a: Check and install missing packages ──────────────────────────────
# The RHOAI PyTorch workbench image does NOT pre-install ultralytics or onnxruntime.
# This cell checks each package first — only downloads what is actually missing.
# Safe to re-run. First run: ~60 s. Subsequent runs: <2 s (nothing to install).
import subprocess, sys, importlib

# module_name -> pip install spec (versions confirmed available on this cluster)
REQUIRED = {
    'ultralytics' : 'ultralytics==8.4.96',
    'onnxruntime' : 'onnxruntime==1.25.0',
}

needs_install = []
for module, pip_spec in REQUIRED.items():
    try:
        importlib.import_module(module)
        print(f'  ✅ {module} already installed — skipping')
    except ImportError:
        print(f'  ⬇️  {module} not found — will install {pip_spec}')
        needs_install.append(pip_spec)

if needs_install:
    print(f'\nInstalling {len(needs_install)} missing package(s)...')
    for pkg in needs_install:
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir', pkg],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            print(f'  ✅ {pkg} installed')
        else:
            print(f'  ❌ {pkg} FAILED:\n{r.stderr[-500:]}')
else:
    print('All required packages already present — nothing to install.')

print('\n✅ Package check complete')

In [ ]:
# ── Cell 0b: Sync lab materials from GitHub ───────────────────────────────────
import subprocess, os, pathlib

REPO_URL   = 'https://github.com/faheemshai/1512_model_training.git'
LOCAL_PATH = os.path.expanduser('~/lab-materials')

if pathlib.Path(LOCAL_PATH, '.git').is_dir():
    r = subprocess.run(['git', '-C', LOCAL_PATH, 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print('Repo up to date:', r.stdout.strip() or 'Already up to date.')
else:
    print('Cloning lab repo (first time, ~10 s)...')
    r = subprocess.run(['git', 'clone', REPO_URL, LOCAL_PATH],
                       capture_output=True, text=True)
    print(r.stderr.strip())

LAB = LOCAL_PATH
print(f'✅ Lab materials ready at: {LAB}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import os

EPOCHS           = 30        # solid convergence on CPU
IMG_SIZE         = 96        # larger than parcel lab — PCB detail matters
BATCH_SIZE       = 16
MODEL_NAME       = 'defect-classifier'
IMAGES_PER_CLASS = 60        # 48 train + 12 val per class

# ── CRITICAL: CLASSES must match YOLOv8 alphabetical sort order ──────────────
# YOLOv8 always sorts class names alphabetically when reading the dataset dir.
# ONNX output: index 0 = contamination, 1 = crack, 2 = pass, 3 = scratch
# This list MUST match that order or the backend maps indices wrong.
CLASSES = sorted(['contamination', 'crack', 'pass', 'scratch'])
print(f'Class order (alphabetical = ONNX output order): {CLASSES}')

# S3 config — read from environment (injected by workbench data connection)
S3_ENDPOINT   = os.environ.get('AWS_S3_ENDPOINT',       'http://s3.openshift-storage.svc:80')
S3_ACCESS_KEY = os.environ.get('AWS_ACCESS_KEY_ID',     '')
S3_SECRET_KEY = os.environ.get('AWS_SECRET_ACCESS_KEY', '')
S3_BUCKET     = os.environ.get('AWS_S3_BUCKET',         '')
NAMESPACE     = open('/var/run/secrets/kubernetes.io/serviceaccount/namespace').read().strip()

S3_MODEL_KEY  = f'models/{MODEL_NAME}/1/model.onnx'

print(f'Namespace  : {NAMESPACE}')
print(f'S3 bucket  : {S3_BUCKET}')
print(f'Model key  : {S3_MODEL_KEY}')
print(f'Epochs     : {EPOCHS}  |  Images/class: {IMAGES_PER_CLASS}  |  IMG_SIZE: {IMG_SIZE}')

In [ ]:
# ── Build a rich 4-class dataset from real PCB seed images ────────────────────
#
# Each class uses one real photo from the git repo as its seed.
# Augmentation pipeline (strong, designed for texture-heavy PCB images):
#   - Horizontal + vertical flips
#   - Random rotation  ±20°
#   - Brightness / contrast / colour / sharpness jitter
#   - Perspective warp (simulates different camera angles)
#   - Random crop + pad (simulates partial field of view)
#   - Gaussian pixel noise (simulates sensor noise)

import pathlib, random
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

DATASET_DIR = pathlib.Path('defect-dataset')
IMG_DIR     = pathlib.Path(LAB) / 'sample-images'

random.seed(42)
np.random.seed(42)

# ── Verify all 4 seed images exist ───────────────────────────────────────────
SEED_FILES = {
    'contamination' : 'contamination.jpg',
    'crack'         : 'crack.jpg',
    'pass'          : 'pass.png',
    'scratch'       : 'scratch.jpg',
}

CLASS_SEEDS = {}
for cls, fname in SEED_FILES.items():
    p = IMG_DIR / fname
    assert p.exists(), f'Missing seed image: {p}'
    CLASS_SEEDS[cls] = Image.open(p).convert('RGB')
    print(f'✅ {cls:<16} ← {fname}  ({p.stat().st_size:,} bytes)')

print('\n✅ All 4 seed images loaded')

In [ ]:
# ── Augmentation function ─────────────────────────────────────────────────────

def augment(img, rng):
    """Strong augmentation pipeline tuned for PCB texture images."""
    img = img.convert('RGB')
    w, h = img.size

    # 1. Random horizontal flip
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)

    # 2. Random vertical flip
    if rng.random() > 0.6:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)

    # 3. Random rotation ±20° (PCBs can be at any angle on a conveyor)
    angle = rng.uniform(-20, 20)
    img = img.rotate(angle, expand=False, fillcolor=(20, 60, 20))

    # 4. Brightness jitter [0.5, 1.5] — factory lighting varies
    img = ImageEnhance.Brightness(img).enhance(rng.uniform(0.5, 1.5))

    # 5. Contrast jitter [0.6, 1.5]
    img = ImageEnhance.Contrast(img).enhance(rng.uniform(0.6, 1.5))

    # 6. Colour saturation [0.6, 1.4]
    img = ImageEnhance.Color(img).enhance(rng.uniform(0.6, 1.4))

    # 7. Sharpness [0.4, 2.0] — some cameras are slightly out of focus
    img = ImageEnhance.Sharpness(img).enhance(rng.uniform(0.4, 2.0))

    # 8. Random crop (0–15% margin) then resize back
    margin = int(min(w, h) * rng.uniform(0.0, 0.15))
    if margin > 4:
        img = img.crop((margin, margin, w - margin, h - margin))

    # 9. Gaussian pixel noise ±15 (sensor noise at high camera gain)
    arr = np.array(img).astype(np.int16)
    arr += rng.integers(-15, 15, arr.shape, dtype=np.int16)
    arr  = np.clip(arr, 0, 255).astype(np.uint8)
    img  = Image.fromarray(arr)

    return img

print('✅ Augmentation function defined')

In [ ]:
# ── Generate dataset ──────────────────────────────────────────────────────────
rng = np.random.default_rng(42)

for split in ['train', 'val']:
    for cls in CLASSES:
        (DATASET_DIR / split / cls).mkdir(parents=True, exist_ok=True)

TRAIN_COUNT = int(IMAGES_PER_CLASS * 0.8)  # 48
VAL_COUNT   = IMAGES_PER_CLASS - TRAIN_COUNT  # 12

for cls in CLASSES:
    seed_img = CLASS_SEEDS[cls]
    imgs = [augment(seed_img, rng) for _ in range(IMAGES_PER_CLASS)]
    for i, img in enumerate(imgs[:TRAIN_COUNT]):
        img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS).save(
            DATASET_DIR / 'train' / cls / f'{i:03d}.jpg', quality=95)
    for i, img in enumerate(imgs[TRAIN_COUNT:]):
        img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS).save(
            DATASET_DIR / 'val' / cls / f'{i:03d}.jpg', quality=95)

print('Dataset generated:')
for split in ['train', 'val']:
    for cls in CLASSES:
        count = len(list((DATASET_DIR / split / cls).glob('*.jpg')))
        print(f'  {split}/{cls}: {count} images')

print('\n✅ Dataset ready — 4 classes, real PCB seed images, strong augmentation')

In [ ]:
# ── Fine-tune YOLOv8n-cls ─────────────────────────────────────────────────────
from ultralytics import YOLO
import time, pathlib

print(f'Starting fine-tuning: {EPOCHS} epochs, img_size={IMG_SIZE}, batch={BATCH_SIZE}')
print(f'Classes (sorted): {CLASSES}')
print('⏱  This will take ~5–10 minutes on CPU ...\n')

WEIGHTS = str(pathlib.Path(LAB) / 'models' / 'yolov8n-cls.pt')
model   = YOLO(WEIGHTS)

t0 = time.time()
results = model.train(
    data          = str(DATASET_DIR),
    epochs        = EPOCHS,
    imgsz         = IMG_SIZE,
    batch         = BATCH_SIZE,
    device        = 'cpu',
    project       = 'runs/classify',
    name          = 'defect-ft',
    exist_ok      = True,
    verbose       = False,
    lr0           = 0.01,
    lrf           = 0.1,
    warmup_epochs = 3,
    patience      = 20,
    dropout       = 0.3,
)

elapsed = time.time() - t0
print(f'\n✅ Training complete in {elapsed/60:.1f} minutes')

In [ ]:
# ── Find best.pt, verify class order, validate ────────────────────────────────
import glob as _glob
_candidates = _glob.glob('**/defect-ft/weights/best.pt', recursive=True)
assert _candidates, 'best.pt not found — did training complete?'
BEST_PT = _candidates[0]
print(f'Best checkpoint: {BEST_PT}')

best_model = YOLO(BEST_PT)

# ── CRITICAL: Verify class order YOLOv8 learned ──────────────────────────────
print(f'\nYOLOv8 internal class names : {best_model.names}')
print(f'Expected (alphabetical)      : {{0: "contamination", 1: "crack", 2: "pass", 3: "scratch"}}')
assert best_model.names[0] == 'contamination', f'Class 0 should be contamination, got {best_model.names[0]}'
assert best_model.names[1] == 'crack',         f'Class 1 should be crack, got {best_model.names[1]}'
assert best_model.names[2] == 'pass',          f'Class 2 should be pass, got {best_model.names[2]}'
assert best_model.names[3] == 'scratch',       f'Class 3 should be scratch, got {best_model.names[3]}'
print('✅ Class order verified: [contamination, crack, pass, scratch]')

val_results = best_model.val(data=str(DATASET_DIR), verbose=False)
print(f'\nValidation top-1 accuracy: {val_results.top1*100:.1f}%')
print(f'Validation top-5 accuracy: {val_results.top5*100:.1f}%')

# Quick per-image test on seed images
print('\nPer-image check on training seeds:')
import tempfile, os
for cls, seed_img in CLASS_SEEDS.items():
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tf:
        seed_img.save(tf.name)
        res  = best_model(tf.name, verbose=False)[0]
        pred = res.names[res.probs.top1]
        conf = res.probs.top1conf.item()
        ok   = pred == cls
        print(f'  {cls:<16} -> {pred:<16} ({conf*100:.1f}%)  {"✅" if ok else "❌"}')
        os.unlink(tf.name)

In [ ]:
# ── Export to ONNX ────────────────────────────────────────────────────────────
print('Exporting to ONNX (opset=13) ...')
onnx_path = best_model.export(format='onnx', imgsz=IMG_SIZE, opset=13)
print(f'✅ ONNX model saved: {onnx_path}')

# ── Verify ONNX with onnxruntime ──────────────────────────────────────────────
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
meta = sess.get_modelmeta()
print(f'ONNX inputs  : {[(i.name, i.shape) for i in sess.get_inputs()]}')
print(f'ONNX outputs : {[(o.name, o.shape) for o in sess.get_outputs()]}')
if 'names' in meta.custom_metadata_map:
    print(f'ONNX class metadata: {meta.custom_metadata_map["names"]}')

# Quick inference test on all 4 seed images
ONNX_CLASSES = ['contamination', 'crack', 'pass', 'scratch']  # alphabetical = ONNX index order
print('\nONNX inference verification:')
for cls, seed_img in CLASS_SEEDS.items():
    arr = np.array(seed_img.convert('RGB').resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
    arr = np.expand_dims(np.transpose(arr, (2, 0, 1)), axis=0)
    raw = sess.run(None, {'images': arr})[0][0]
    exp_s = np.exp(raw - raw.max()); probs = exp_s / exp_s.sum()
    pred  = ONNX_CLASSES[int(np.argmax(probs))]
    conf  = float(probs.max())
    ok    = pred == cls
    print(f'  expected={cls:<16} ONNX_pred={pred:<16} conf={conf*100:.1f}%  {"✅" if ok else "❌"}')
    print(f'    logits={[round(float(x),4) for x in raw]}  spread={float(max(raw)-min(raw)):.4f}')

In [ ]:
# ── Upload ONNX model to S3 ───────────────────────────────────────────────────
import boto3
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url          = S3_ENDPOINT,
    aws_access_key_id     = S3_ACCESS_KEY,
    aws_secret_access_key = S3_SECRET_KEY,
    region_name           = 'us-east-1',
    config                = Config(signature_version='s3v4'),
    verify                = False,
)

print(f'Uploading to s3://{S3_BUCKET}/{S3_MODEL_KEY} ...')
s3.upload_file(str(onnx_path), S3_BUCKET, S3_MODEL_KEY)

obj = s3.head_object(Bucket=S3_BUCKET, Key=S3_MODEL_KEY)
size_kb = obj['ContentLength'] / 1024
print(f'✅ Upload complete — {size_kb:.1f} KB at {S3_MODEL_KEY}')
print(f'\nModel path for KServe: models/{MODEL_NAME}/')
print('(KServe storageInitializer downloads prefix → serves /mnt/models/1/model.onnx)')

In [ ]:
# ── Class order confirmation ──────────────────────────────────────────────────
#
# The backend app/base/backend.yaml already has the correct class order hardcoded:
#   CLASSES: contamination,crack,pass,scratch
# Bob (Step 10b) will apply that ConfigMap fresh — no manual patch needed.

CORRECT_CLASSES = 'contamination,crack,pass,scratch'
print(f'✅ ONNX class order confirmed: {CORRECT_CLASSES}')
print('   This matches backend.yaml CLASSES — no manual patch required.')
print('   Bob (Step 10b) will apply the correct ConfigMap automatically.')

In [ ]:
# ── What happens next ─────────────────────────────────────────────────────────
#
# The ONNX model is now in S3 at: models/defect-classifier/1/model.onnx
#
# Step 8 (Bob) will deploy KServe:
#   1. ServingRuntime (ovms-runtime) — tells KServe how to run ONNX models
#   2. InferenceService (defect-classifier) — downloads model from S3, starts OVMS
#   3. ClusterIP service + Route — exposes the endpoint externally
#
# Notebook 4 will then test the live inference endpoint.
# No oc commands are needed from this notebook.

INFERENCE_URL = f'https://defect-classifier-{NAMESPACE}.apps.itz-t53413.hub01-lb.techzone.ibm.com'
print('✅ Model uploaded to S3 — ready for KServe deployment')
print(f'   Future inference URL: {INFERENCE_URL}')
print('   → Follow Step 8 in the lab guide to deploy KServe with Bob')

In [ ]:
# ── Save model-info for downstream notebooks ──────────────────────────────────
import json

class_info = {
    'classes'          : ONNX_CLASSES,
    'model_name'       : MODEL_NAME,
    'model_key'        : S3_MODEL_KEY,
    'bucket'           : S3_BUCKET,
    'img_size'         : IMG_SIZE,
    'epochs'           : EPOCHS,
    'images_per_class' : IMAGES_PER_CLASS,
    'val_top1'         : round(float(val_results.top1), 4),
}
with open('model-info.json', 'w') as f:
    json.dump(class_info, f, indent=2)
print('✅ Saved model-info.json')
print(json.dumps(class_info, indent=2))

print('\n══════════════════════════════════════════════════')
print(' Notebook 3 complete — Manufacturing Defect Model')
print('══════════════════════════════════════════════════')
print(f' ONNX class order : {ONNX_CLASSES}')
print(f' Val top-1 acc    : {val_results.top1*100:.1f}%')
print(f' ONNX path        : {onnx_path}')
print(f' S3 key           : {S3_MODEL_KEY}')
print('\n➡  Next: Run Notebook 4 to test the live inference endpoint')